# Fase 5.4 — Validación con referencia del pipeline completo

**TFM · Restauración y super-resolución de imágenes históricas mediante GAN — Javier Riesco Velasco**

Este notebook cierra el apartado 5.4 de la memoria: la verificación de extremo a extremo
del pipeline definitivo (OE3). Compara dos configuraciones del **mismo pipeline secuencial**
sobre el **mismo conjunto de test con referencia**:

| Configuración | Etapa 1 · inpainting | Etapa 2 · super-resolución ×4 |
|---|---|---|
| **Preentrenada** | LaMa `big-lama` preentrenado | A-ESRGAN `A_ESRGAN_Single.pth` |
| **Ajustada** | LaMa ajustado (5.2, `lama_best.pth`) | A-ESRGAN brazo F_flick, iteración 400 |

### Conjunto de evaluación

Se parte de las **30 imágenes del test sintético de super-resolución** (`Fase1/pares/test/`),
que ya llevan la degradación espectral calibrada de la Fase 4b en su versión LQ, y se les
aplica encima el **daño localizado del módulo de inpainting de la Fase 1**
(`synthetic_degradation`, la misma `DamageConfig.vintage_base()` con la que se generó el
conjunto de LaMa). El resultado es un trío por imagen:

- `entrada` — LR con degradación espectral **y** daño localizado (roturas, arañazos, manchas),
- `mascara` — máscara binaria del daño, a resolución LR,
- `referencia` — la HR limpia original, intacta.

El daño se pinta sobre la LR y no sobre la HR porque es el orden real del pipeline: el sistema
recibe una fotografía de baja calidad ya dañada, la inpinta a su resolución nativa y después la
amplía ×4. La máscara se propaga ×4 por vecino más próximo para medir sobre la región
reconstruida en la salida final.

### Pregunta que responde el apartado

Más allá de sumar 5.2 y 5.3: **¿se conserva al encadenar la mejora que cada etapa consigue por
separado, o la ampliación ×4 amplifica los artefactos que el inpainting introduce en la región
reconstruida?** Por eso las métricas se reportan también restringidas a la región de máscara y
como trayectoria por etapas, no solo como cifra final.

> **Advertencia de comparabilidad (repetir en la memoria).** Las cifras de este notebook se
> obtienen con el módulo de degradación espectral calibrado y con daño de inpainting encima.
> **No son comparables** con la línea base del apartado 5.1 (PSNR 18,52 dB, SSIM 0,767), medida
> con el pipeline de degradación de la tercera entrega y sobre una única imagen con máscara
> rectangular. Las dos columnas se llaman «preentrenado» y tientan a restarse: no se restan.

> **Qué no va aquí.** Ningún material histórico real. La evaluación NIQE sobre las 28
> fotografías reales es el capítulo 6 completo.

## 1. Entorno, dependencias y rutas

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import subprocess, sys

# simple-lama-inpainting va SIN dependencias: sus pines (numpy<2, pillow<10) están obsoletos y
# degradarían el entorno. En ejecución solo necesita torch, numpy, cv2 y PIL, que Colab ya trae
# (el paquete 'fire' que declara es para su CLI, que este notebook no usa).
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
                'simple-lama-inpainting', 'lpips'], check=True)

# basicsr sí puede resolver dependencias: no toca numpy.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'basicsr>=1.3.3.11'], check=True)

import numpy, PIL
print('numpy', numpy.__version__, '| Pillow', PIL.__version__)   # deben ser 2.x y 11.x

In [ ]:
# Dependencias. simple-lama-inpainting trae LaMa; basicsr trae la arquitectura RRDB de A-ESRGAN.
#%pip install -q simple-lama-inpainting "basicsr>=1.3.3.11" lpips scikit-image pandas

import site, sys
from pathlib import Path

# Parche basicsr: torchvision >= 0.16 eliminó functional_tensor.
for base in site.getsitepackages() + [site.getusersitepackages()]:
    p = Path(base) / 'basicsr' / 'data' / 'degradations.py'
    if p.exists():
        t = p.read_text()
        viejo = 'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
        nuevo = 'from torchvision.transforms.functional import rgb_to_grayscale'
        if viejo in t:
            p.write_text(t.replace(viejo, nuevo))
            print('Parche basicsr aplicado:', p)

# Parche PIL._typing: Pillow 12 eliminó _Ink, que torchvision aún espera.
try:
    import PIL._typing
    if not hasattr(PIL._typing, '_Ink'):
        from typing import Union
        PIL._typing._Ink = Union[int, tuple]
except ModuleNotFoundError:
    pass   # Pillow anterior a la 10: no hay nada que parchear

print('Dependencias y parches listos.')

In [ ]:
import gc, json, shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageFile
from scipy.stats import wilcoxon

ImageFile.LOAD_TRUNCATED_IMAGES = True

# ── Paleta Okabe-Ito (misma que en Fase 4d, para coherencia de figuras) ──────
OI_BLUE   = '#0072B2'
OI_ORANGE = '#E69F00'
OI_GREEN  = '#009E73'
OI_PINK   = '#CC79A7'
OI_GREY   = '#999999'

matplotlib.rcParams.update({
    'font.family':     'sans-serif',
    'font.size':       9,
    'axes.labelsize':  9,
    'axes.titlesize':  9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi':      150,
    'savefig.dpi':     300,
    'savefig.bbox':    'tight',
})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def torch_load(ruta, map_location='cpu'):
    """torch.load compatible con torch >= 2.6, donde weights_only pasó a True por defecto.

    Los checkpoints de BasicSR y el del ajuste fino de LaMa son diccionarios pickled
    con objetos que el cargador restringido rechaza.
    """
    try:
        return torch.load(ruta, map_location=map_location, weights_only=False)
    except TypeError:                      # torch < 1.13 no conoce el argumento
        return torch.load(ruta, map_location=map_location)


print('torch:', torch.__version__, '| device:', DEVICE,
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'sin GPU')

In [ ]:
# ── Rutas ────────────────────────────────────────────────────────────────────
PROJECT_DIR = Path('/content/drive/MyDrive/TFM')
FASE1       = PROJECT_DIR / 'Fase1'
FASE4B      = PROJECT_DIR / 'Fase4b'
FASE4C      = PROJECT_DIR / 'Fase4c'
PESOS_DIR   = PROJECT_DIR / 'pesos'
FASE5_4     = PROJECT_DIR / 'Fase5_4'

# Test sintético de super-resolución (30 pares LQ/HR con degradación espectral).
DIR_TEST_LR = FASE1 / 'pares' / 'test' / 'lr'
DIR_TEST_HR = FASE1 / 'pares' / 'test' / 'hr'

# Checkpoints.
PTH_AESRGAN_PRE = PESOS_DIR / 'A_ESRGAN_Single.pth'
PTH_AESRGAN_FT  = FASE4C / 'checkpoints' / 'brazoF_flick' / 'models' / 'net_g_400.pth'
PTH_LAMA_FT     = PROJECT_DIR / 'checkpoints' / 'lama_finetuned' / 'lama_best.pth'

# Salidas del apartado 5.4.
DIR_ENTRADA     = FASE5_4 / 'entrada'        # LR espectral + daño localizado
DIR_MASCARA     = FASE5_4 / 'mascaras'       # máscara binaria a resolución LR
DIR_ENMASCARADA = FASE5_4 / 'enmascarada'    # entrada con el agujero a negro (etapa 0)
DIR_LAMA_PRE    = FASE5_4 / 'lama_pre'
DIR_LAMA_FT     = FASE5_4 / 'lama_ft'
DIR_SR_PRE      = FASE5_4 / 'sr_pipeline_pre'
DIR_SR_FT       = FASE5_4 / 'sr_pipeline_ft'
DIR_METRICAS    = FASE5_4 / 'metricas'
DIR_FIGURAS     = FASE5_4 / 'figuras'

for d in (DIR_ENTRADA, DIR_MASCARA, DIR_ENMASCARADA, DIR_LAMA_PRE, DIR_LAMA_FT,
          DIR_SR_PRE, DIR_SR_FT, DIR_METRICAS, DIR_FIGURAS):
    d.mkdir(parents=True, exist_ok=True)

print('Comprobación de rutas de entrada:')
for p, etiq in [(DIR_TEST_LR, 'test LR (espectral)'),
                (DIR_TEST_HR, 'test HR (referencia)'),
                (PTH_AESRGAN_PRE, 'A-ESRGAN preentrenado'),
                (PTH_AESRGAN_FT, 'A-ESRGAN F_flick iter 400'),
                (PTH_LAMA_FT, 'LaMa ajustado'),
                (PROJECT_DIR / 'synthetic_degradation', 'paquete synthetic_degradation')]:
    print(f'  {"✓" if p.exists() else "✗ FALTA"}  {etiq}: {p}')

print(f'\nSalidas en: {FASE5_4}')

## 2. Construcción del conjunto de evaluación

Se aplica el daño localizado de la Fase 1 sobre cada LR del test sintético. Dos decisiones
que conviene dejar escritas porque condicionan la validez de la comparación:

1. **Solo se toman del generador los píxeles del interior de la máscara.** El paquete
   `synthetic_degradation` produce el trío (degradada, GT vintage, máscara) aplicando también
   el toneado global de época; aquí ese toneado no interesa, porque desplazaría el color de la
   entrada respecto a la referencia HR y contaminaría las métricas. Se compone
   `entrada = LR original fuera de la máscara, degradada del generador dentro`. Como LaMa
   descarta el contenido bajo la máscara, lo único que aporta el generador es la **geometría
   del daño**, que es exactamente lo que se quiere reutilizar de 5.2.
2. **Semilla fija y derivada del índice de la imagen.** Las dos configuraciones del pipeline
   reciben byte a byte la misma entrada y la misma máscara, requisito del test pareado de
   Wilcoxon.

In [ ]:
import sys
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from synthetic_degradation import DamageConfig, make_training_sample, mask_coverage

CFG_DANIO = DamageConfig.vintage_base()   # la misma configuración que generó el dataset de LaMa
SEED_BASE = 20250904                      # semilla fija: entradas idénticas para ambos pipelines

print('Configuración de daño (idéntica a la del apartado 5.2):')
print(' ', CFG_DANIO)

In [ ]:
EXTS = ('*.png', '*.jpg', '*.jpeg')

def listar(dirs):
    return sorted(p for ext in EXTS for p in Path(dirs).glob(ext))

rutas_lr = listar(DIR_TEST_LR)
assert rutas_lr, f'No hay imágenes en {DIR_TEST_LR}'
print(f'{len(rutas_lr)} imágenes LR en el test sintético.')

registros_conjunto = []

for i, ruta_lr in enumerate(rutas_lr):
    stem = ruta_lr.stem
    ruta_hr = next((DIR_TEST_HR / f'{stem}{s}' for s in ('.png', '.jpg', '.jpeg')
                    if (DIR_TEST_HR / f'{stem}{s}').exists()), None)
    assert ruta_hr is not None, f'Falta la referencia HR de {stem} en {DIR_TEST_HR}'

    lr_bgr = cv2.imread(str(ruta_lr), cv2.IMREAD_COLOR)
    lr_rgb = cv2.cvtColor(lr_bgr, cv2.COLOR_BGR2RGB)

    # Daño localizado con la misma configuración de la Fase 1.
    rng = np.random.default_rng(SEED_BASE + i)
    deg_pkg, _gt_pkg, mask = make_training_sample(lr_rgb, rng, CFG_DANIO)

    # El generador podría devolver otro tamaño según su configuración: se realinea a la LR.
    if deg_pkg.shape[:2] != lr_rgb.shape[:2]:
        h, w = lr_rgb.shape[:2]
        if i == 0:
            print(f'[AVISO] El generador devuelve {deg_pkg.shape[:2]}; se realinea a {(h, w)}.')
        deg_pkg = cv2.resize(deg_pkg, (w, h), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

    mask = np.where(mask > 127, 255, 0).astype(np.uint8)
    assert set(np.unique(mask)).issubset({0, 255}), f'{stem}: máscara no binaria'

    # Composición: fuera de la máscara se conserva la LR original intacta.
    m3 = (mask > 127)[..., None]
    entrada_rgb = np.where(m3, deg_pkg, lr_rgb).astype(np.uint8)

    # Invariante: la degradación espectral de la Fase 4b no se toca fuera del daño.
    assert np.array_equal(entrada_rgb[mask == 0], lr_rgb[mask == 0]), \
        f'{stem}: la entrada difiere de la LR fuera de la máscara'

    # Entrada con el agujero a negro: es la etapa 0 de la trayectoria de métricas.
    enmascarada_rgb = entrada_rgb.copy()
    enmascarada_rgb[mask > 127] = 0

    cv2.imwrite(str(DIR_ENTRADA / f'{stem}.png'),
                cv2.cvtColor(entrada_rgb, cv2.COLOR_RGB2BGR))
    cv2.imwrite(str(DIR_ENMASCARADA / f'{stem}.png'),
                cv2.cvtColor(enmascarada_rgb, cv2.COLOR_RGB2BGR))
    cv2.imwrite(str(DIR_MASCARA / f'{stem}.png'), mask)

    registros_conjunto.append({
        'imagen': stem,
        'ruta_hr': str(ruta_hr),
        'alto_lr': lr_rgb.shape[0], 'ancho_lr': lr_rgb.shape[1],
        'cobertura': float(mask_coverage(mask)),
    })

df_conjunto = pd.DataFrame(registros_conjunto).set_index('imagen')
cob = df_conjunto['cobertura']
print(f'\nConjunto 5.4 generado: {len(df_conjunto)} tríos en {FASE5_4}')
print(f'Tamaño LR: {df_conjunto["ancho_lr"].min()}×{df_conjunto["alto_lr"].min()} '
      f'a {df_conjunto["ancho_lr"].max()}×{df_conjunto["alto_lr"].max()} px')
print(f'Cobertura de daño — min {cob.min():.2%} | media {cob.mean():.2%} | max {cob.max():.2%}')

if cob.min() == 0:
    print('[AVISO] Alguna imagen quedó sin daño: revisa la configuración antes de medir.')
if cob.max() > 0.5:
    print('[AVISO] Cobertura muy alta en alguna imagen: el daño domina la escena.')

df_conjunto.to_csv(DIR_METRICAS / 'conjunto_5_4.csv')

In [ ]:
# Inspección visual del conjunto generado (3 ejemplos).
muestras = df_conjunto.index[:3]
fig, axes = plt.subplots(len(muestras), 4, figsize=(11, 2.9 * len(muestras)))
if len(muestras) == 1:
    axes = axes[np.newaxis, :]

cabeceras = ['LR espectral (Fase 4b)', 'Entrada 5.4 (LR + daño)', 'Máscara', 'Referencia HR']
for col, cab in enumerate(cabeceras):
    axes[0, col].set_title(cab, fontsize=8, fontweight='bold', pad=3)

for fila, stem in enumerate(muestras):
    ruta_lr_muestra = next(p for p in rutas_lr if p.stem == stem)
    lr   = cv2.cvtColor(cv2.imread(str(ruta_lr_muestra)), cv2.COLOR_BGR2RGB)
    ent  = cv2.cvtColor(cv2.imread(str(DIR_ENTRADA / f'{stem}.png')), cv2.COLOR_BGR2RGB)
    msk  = cv2.imread(str(DIR_MASCARA / f'{stem}.png'), cv2.IMREAD_GRAYSCALE)
    hr   = cv2.cvtColor(cv2.imread(df_conjunto.loc[stem, 'ruta_hr']), cv2.COLOR_BGR2RGB)
    for col, (img, cmap) in enumerate([(lr, None), (ent, None), (msk, 'gray'), (hr, None)]):
        axes[fila, col].imshow(img, cmap=cmap)
        axes[fila, col].axis('off')
    axes[fila, 0].text(0.02, 0.03, stem[:18], transform=axes[fila, 0].transAxes,
                       fontsize=7, color='white',
                       bbox=dict(facecolor='black', alpha=0.5, pad=1, edgecolor='none'))

plt.suptitle('Conjunto de evaluación de 5.4 — degradación espectral más daño localizado',
             fontsize=9, fontweight='bold', y=0.995)
plt.subplots_adjust(wspace=0.02, hspace=0.05, top=0.94)
ruta_fig = DIR_FIGURAS / 'conjunto_5_4_ejemplos.png'
fig.savefig(ruta_fig, dpi=200, bbox_inches='tight')
plt.show()
print('Figura guardada:', ruta_fig)

## 3. Etapa 1 — inpainting con LaMa

Ambos modelos LaMa se ejecutan sobre la **misma** entrada y la **misma** máscara. El ajustado
carga el `state_dict` del checkpoint de 5.2 sobre el módulo JIT de `SimpleLama`, igual que en el
notebook de ajuste fino. La salida de `SimpleLama` viene rellenada a múltiplo de 8, así que se
recorta al tamaño original en lugar de reescalarla: reescalar introduciría un remuestreo que no
forma parte del pipeline.

In [ ]:
from simple_lama_inpainting import SimpleLama

class EjecutorLaMa:
    """Envuelve SimpleLama para reutilizar el modelo entre imágenes.

    ckpt=None -> pesos big-lama preentrenados.
    ckpt=ruta -> state_dict del ajuste fino del apartado 5.2.
    """
    def __init__(self, ckpt=None):
        self.lama = SimpleLama()
        if ckpt is not None:
            estado = torch_load(ckpt, map_location='cpu')
            info = self.lama.model.load_state_dict(estado, strict=False)
            print(f'  checkpoint cargado ({Path(ckpt).name}) — '
                  f'{len(info.missing_keys)} claves ausentes, '
                  f'{len(info.unexpected_keys)} inesperadas')
        self.lama.model.to(DEVICE).eval()

    def __call__(self, img: Image.Image, mask: Image.Image) -> Image.Image:
        out = self.lama(img, mask).convert('RGB')
        if out.size != img.size:          # SimpleLama rellena a múltiplo de 8
            out = out.crop((0, 0, img.size[0], img.size[1]))
        return out

    def liberar(self):
        del self.lama
        gc.collect()
        torch.cuda.empty_cache()


def inpaintar_conjunto(ejecutor, dir_salida):
    dir_salida = Path(dir_salida)
    dir_salida.mkdir(parents=True, exist_ok=True)
    for stem in df_conjunto.index:
        img  = Image.open(DIR_ENTRADA / f'{stem}.png').convert('RGB')
        mask = Image.open(DIR_MASCARA / f'{stem}.png').convert('L')
        ejecutor(img, mask).save(dir_salida / f'{stem}.png')
    print(f'  {len(df_conjunto)} imágenes → {dir_salida}')

print('LaMa preentrenado:')
lama_pre = EjecutorLaMa(None)
inpaintar_conjunto(lama_pre, DIR_LAMA_PRE)
lama_pre.liberar(); del lama_pre

print('LaMa ajustado (5.2):')
assert PTH_LAMA_FT.exists(), f'No encuentro el checkpoint de LaMa ajustado en {PTH_LAMA_FT}'
lama_ft = EjecutorLaMa(PTH_LAMA_FT)
inpaintar_conjunto(lama_ft, DIR_LAMA_FT)
lama_ft.liberar(); del lama_ft

print('\nEtapa de inpainting completada para las dos configuraciones.')

## 4. Etapa 2 — super-resolución ×4 con A-ESRGAN

Se reutiliza sin cambios la función de inferencia por tiles de la Fase 4d, de modo que la
etapa de super-resolución de este apartado sea idéntica a la que se validó de forma aislada en
5.3. La entrada de esta etapa es la salida de LaMa correspondiente a cada configuración: el
pipeline preentrenado consume la salida del LaMa preentrenado y el ajustado la del LaMa ajustado.

In [ ]:
from basicsr.archs.rrdbnet_arch import RRDBNet

def cargar_modelo_sr(pth_path):
    """Carga un checkpoint RRDB en el dispositivo activo y lo pone en modo eval."""
    net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                  num_block=23, num_grow_ch=32, scale=4)
    ckpt = torch_load(pth_path, map_location='cpu')
    estado = ckpt.get('params_ema', ckpt.get('params', ckpt))
    net.load_state_dict(estado, strict=True)
    return net.eval().to(DEVICE)


def _inferir_tiles(modelo, t, tile, tile_pad):
    """Inferencia por tiles con solapamiento para evitar OOM y artefactos de borde."""
    b, c, h, w = t.shape
    escala = 4
    out = torch.zeros(b, c, h * escala, w * escala, device=t.device)
    tiles_h = max(1, (h + tile - 1) // tile)
    tiles_w = max(1, (w + tile - 1) // tile)
    for i in range(tiles_h):
        for j in range(tiles_w):
            y0 = max(0, i * tile - tile_pad); y1 = min(h, (i + 1) * tile + tile_pad)
            x0 = max(0, j * tile - tile_pad); x1 = min(w, (j + 1) * tile + tile_pad)
            sr_patch = modelo(t[:, :, y0:y1, x0:x1])
            oy0 = (y0 + (tile_pad if i > 0 else 0)) * escala
            oy1 = (y1 - (tile_pad if i < tiles_h - 1 else 0)) * escala
            ox0 = (x0 + (tile_pad if j > 0 else 0)) * escala
            ox1 = (x1 - (tile_pad if j < tiles_w - 1 else 0)) * escala
            py0 = (tile_pad if i > 0 else 0) * escala; py1 = py0 + (oy1 - oy0)
            px0 = (tile_pad if j > 0 else 0) * escala; px1 = px0 + (ox1 - ox0)
            out[:, :, oy0:oy1, ox0:ox1] = sr_patch[:, :, py0:py1, px0:px1]
    return out


def superresolver(modelo, dir_lr, dir_salida, tile=512, tile_pad=32):
    dir_salida = Path(dir_salida); dir_salida.mkdir(parents=True, exist_ok=True)
    with torch.no_grad():
        for stem in df_conjunto.index:
            img_bgr = cv2.imread(str(Path(dir_lr) / f'{stem}.png'), cv2.IMREAD_COLOR)
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.
            t = torch.from_numpy(img_rgb.transpose(2, 0, 1)).unsqueeze(0).to(DEVICE)
            out = _inferir_tiles(modelo, t, tile, tile_pad)
            sr = (out.squeeze(0).permute(1, 2, 0).cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
            cv2.imwrite(str(dir_salida / f'{stem}.png'), cv2.cvtColor(sr, cv2.COLOR_RGB2BGR))
    print(f'  {len(df_conjunto)} imágenes → {dir_salida}')


print('A-ESRGAN preentrenado sobre la salida del LaMa preentrenado:')
modelo_sr_pre = cargar_modelo_sr(PTH_AESRGAN_PRE)
superresolver(modelo_sr_pre, DIR_LAMA_PRE, DIR_SR_PRE)
del modelo_sr_pre; gc.collect(); torch.cuda.empty_cache()

print('A-ESRGAN brazo F_flick (iter 400) sobre la salida del LaMa ajustado:')
modelo_sr_ft = cargar_modelo_sr(PTH_AESRGAN_FT)
superresolver(modelo_sr_ft, DIR_LAMA_FT, DIR_SR_FT)
del modelo_sr_ft; gc.collect(); torch.cuda.empty_cache()

print('\nPipeline completo ejecutado en las dos configuraciones.')

## 5. Protocolo de medida

Se mantiene el protocolo definido en 5.1 y usado en 5.3: **PSNR, SSIM y LPIPS**, calculados
**global** y **restringidos a la región de máscara**, contra la referencia HR limpia.

Tres convenciones que hay que declarar en la memoria:

- **Escala de la referencia.** Cada predicción se compara contra la referencia HR remuestreada
  al tamaño de esa predicción. Las etapas intermedias (enmascarada y salida de LaMa) viven a
  resolución LR, así que su referencia es la HR reducida; la salida final vive a ×4 y se compara
  contra la HR a tamaño nativo. La trayectoria por etapas es por tanto **indicativa del
  comportamiento relativo entre configuraciones**, no una serie de cifras comparables entre sí:
  el cambio de escala de la referencia altera el valor absoluto de las tres métricas.
- **Región de máscara.** La máscara se remuestrea por vecino más próximo al tamaño de la
  predicción. PSNR y SSIM de máscara se calculan sobre los píxeles marcados (SSIM promediando el
  mapa local de similitud dentro de la máscara). LPIPS es una métrica de parche y no admite
  píxeles sueltos: se calcula sobre el **recorte del rectángulo envolvente** del daño, con un
  mínimo de 32 px de lado.
- **Comparación pareada.** Las dos configuraciones ven la misma entrada, así que todas las
  comparaciones son por imagen y el contraste estadístico es pareado.

In [ ]:
import lpips
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

lpips_fn = lpips.LPIPS(net='alex').eval().to(DEVICE)
LPIPS_LADO_MIN = 32   # lado mínimo del recorte para que LPIPS sea estable


def _a_tensor_lpips(img_rgb_uint8):
    t = torch.from_numpy(img_rgb_uint8.astype(np.float32) / 255.).permute(2, 0, 1).unsqueeze(0)
    return t * 2 - 1                        # LPIPS espera [-1, 1]


_LPIPS_EN_CPU = False    # se activa si la GPU se queda sin memoria con las salidas ×4


def lpips_par(a_rgb, b_rgb):
    """LPIPS entre dos arrays uint8 RGB, con caída a CPU ante falta de memoria en GPU."""
    global _LPIPS_EN_CPU
    ta, tb = _a_tensor_lpips(a_rgb), _a_tensor_lpips(b_rgb)
    if not _LPIPS_EN_CPU:
        try:
            with torch.no_grad():
                return float(lpips_fn(ta.to(DEVICE), tb.to(DEVICE)).item())

        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            lpips_fn.cpu()
            _LPIPS_EN_CPU = True
            print('[AVISO] LPIPS pasa a CPU: memoria de GPU insuficiente para el tamaño ×4.')
    with torch.no_grad():
        return float(lpips_fn(ta, tb).item())


def _bbox_mascara(mask_bool, lado_min=LPIPS_LADO_MIN):
    """Rectángulo envolvente del daño, expandido hasta 'lado_min' y recortado a la imagen."""
    if not mask_bool.any():
        return None
    filas = np.where(mask_bool.any(axis=1))[0]
    cols  = np.where(mask_bool.any(axis=0))[0]
    y0, y1 = int(filas[0]), int(filas[-1]) + 1
    x0, x1 = int(cols[0]),  int(cols[-1])  + 1
    h, w = mask_bool.shape
    if (y1 - y0) < lado_min:
        cy = (y0 + y1) // 2
        y0 = max(0, cy - lado_min // 2); y1 = min(h, y0 + lado_min); y0 = max(0, y1 - lado_min)
    if (x1 - x0) < lado_min:
        cx = (x0 + x1) // 2
        x0 = max(0, cx - lado_min // 2); x1 = min(w, x0 + lado_min); x0 = max(0, x1 - lado_min)
    if (y1 - y0) < lado_min or (x1 - x0) < lado_min:
        return None                        # imagen más pequeña que el recorte mínimo
    return y0, y1, x0, x1


def metricas_par(pred_rgb, ref_rgb, mask_bool):
    """PSNR, SSIM y LPIPS, global y en región de máscara. Arrays uint8 RGB del mismo tamaño."""
    psnr = float(peak_signal_noise_ratio(ref_rgb, pred_rgb, data_range=255))
    ssim, ssim_map = structural_similarity(ref_rgb, pred_rgb, channel_axis=2,
                                           data_range=255, full=True)
    fila = {'PSNR': psnr, 'SSIM': float(ssim), 'LPIPS': lpips_par(pred_rgb, ref_rgb)}

    if mask_bool.any():
        err = pred_rgb[mask_bool].astype(np.float64) - ref_rgb[mask_bool].astype(np.float64)
        mse_m = float(np.mean(err ** 2))
        fila['PSNR_mask'] = float('inf') if mse_m == 0 else float(20 * np.log10(255.0 / np.sqrt(mse_m)))
        mapa = ssim_map.mean(axis=-1) if ssim_map.ndim == 3 else ssim_map
        fila['SSIM_mask'] = float(mapa[mask_bool].mean())
        bbox = _bbox_mascara(mask_bool)
        if bbox is not None:
            y0, y1, x0, x1 = bbox
            fila['LPIPS_mask'] = lpips_par(np.ascontiguousarray(pred_rgb[y0:y1, x0:x1]),
                                           np.ascontiguousarray(ref_rgb[y0:y1, x0:x1]))
        else:
            fila['LPIPS_mask'] = np.nan
    else:
        fila.update({'PSNR_mask': np.nan, 'SSIM_mask': np.nan, 'LPIPS_mask': np.nan})
    return fila


def _leer_rgb(ruta):
    img = cv2.imread(str(ruta), cv2.IMREAD_COLOR)
    assert img is not None, f'No se pudo leer {ruta}'
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def _alinear(img_rgb, tam_wh, interp=cv2.INTER_LANCZOS4):
    w, h = tam_wh
    if img_rgb.shape[1] == w and img_rgb.shape[0] == h:
        return img_rgb
    return cv2.resize(img_rgb, (w, h), interpolation=interp)


print('Protocolo de medida definido. LPIPS en', DEVICE)

In [ ]:
# ── Métricas por imagen, por etapa y por configuración ───────────────────────
ETAPAS = {
    'Enmascarada':  {'pre': DIR_ENMASCARADA, 'ft': DIR_ENMASCARADA},   # etapa común a ambas
    'LaMa':         {'pre': DIR_LAMA_PRE,    'ft': DIR_LAMA_FT},
    'A-ESRGAN ×4':  {'pre': DIR_SR_PRE,      'ft': DIR_SR_FT},
}
CONFIGS = {'pre': 'Preentrenado', 'ft': 'Ajustado'}

registros = []
for stem in df_conjunto.index:
    hr   = _leer_rgb(df_conjunto.loc[stem, 'ruta_hr'])
    mask = cv2.imread(str(DIR_MASCARA / f'{stem}.png'), cv2.IMREAD_GRAYSCALE)
    for etapa, dirs in ETAPAS.items():
        for cfg, dir_cfg in dirs.items():
            pred = _leer_rgb(Path(dir_cfg) / f'{stem}.png')
            tam  = (pred.shape[1], pred.shape[0])
            ref  = _alinear(hr, tam)
            msk  = _alinear(mask, tam, interp=cv2.INTER_NEAREST) > 127
            fila = metricas_par(pred, ref, msk)
            fila.update({'imagen': stem, 'etapa': etapa,
                         'config': CONFIGS[cfg], 'clave_config': cfg,
                         'ancho': tam[0], 'alto': tam[1]})
            registros.append(fila)

df_met = pd.DataFrame(registros)
df_met.to_csv(DIR_METRICAS / 'metricas_por_imagen_5_4.csv', index=False)
print('Métricas por imagen guardadas:', DIR_METRICAS / 'metricas_por_imagen_5_4.csv')
print(f'{len(df_met)} filas = {len(df_conjunto)} imágenes × {len(ETAPAS)} etapas × 2 configuraciones')
df_met.head()

## 6. Resultados

### 6.1 Trayectoria por etapas y tabla del pipeline completo

In [ ]:
METRICAS = ['PSNR', 'SSIM', 'LPIPS', 'PSNR_mask', 'SSIM_mask', 'LPIPS_mask']
ORDEN_ETAPAS = list(ETAPAS.keys())

resumen = (df_met.groupby(['etapa', 'config'])[METRICAS]
                 .mean()
                 .reindex(pd.MultiIndex.from_product([ORDEN_ETAPAS, list(CONFIGS.values())],
                                                     names=['etapa', 'config'])))
print('Trayectoria de métricas por etapas (media sobre n = %d)' % len(df_conjunto))
print('Recordatorio: la referencia cambia de escala entre las dos primeras etapas y la tercera.\n')
print(resumen.to_string(float_format='{:.4f}'.format))
resumen.to_csv(DIR_METRICAS / 'trayectoria_etapas_5_4.csv')

In [ ]:
# Tabla principal del apartado: pipeline completo, preentrenado vs ajustado.
final = df_met[df_met['etapa'] == 'A-ESRGAN ×4']
piv   = final.pivot_table(index='config', values=METRICAS, aggfunc='mean').loc[list(CONFIGS.values())]

tabla_final = pd.DataFrame({
    'Métrica':      ['LPIPS ↓', 'PSNR ↑ (dB)', 'SSIM ↑',
                     'LPIPS máscara ↓', 'PSNR máscara ↑ (dB)', 'SSIM máscara ↑'],
    'Preentrenado': [piv.loc['Preentrenado', m] for m in
                     ['LPIPS', 'PSNR', 'SSIM', 'LPIPS_mask', 'PSNR_mask', 'SSIM_mask']],
    'Ajustado':     [piv.loc['Ajustado', m] for m in
                     ['LPIPS', 'PSNR', 'SSIM', 'LPIPS_mask', 'PSNR_mask', 'SSIM_mask']],
})
tabla_final['Δ (ajustado − preentrenado)'] = tabla_final['Ajustado'] - tabla_final['Preentrenado']

print('Pipeline completo sobre el test sintético con daño de inpainting '
      f'(n = {len(df_conjunto)})\n')
print(tabla_final.to_string(index=False, float_format='{:.4f}'.format))
tabla_final.to_csv(DIR_METRICAS / 'resumen_pipeline_5_4.csv', index=False)

### 6.2 Contraste estadístico

Wilcoxon de rangos con signo sobre los pares por imagen, con **corrección de Bonferroni**: la
familia son las seis comparaciones de la tabla anterior (tres métricas × dos regiones) sobre el
mismo conjunto, de modo que el umbral corregido es α = 0,05 / 6 ≈ 0,0083. Se reporta el valor
`p` sin corregir y el `p` ajustado, junto con la mediana de la diferencia y el número de imágenes
que mejoran, porque con n = 30 la significación por sí sola dice poco sobre la magnitud.

In [ ]:
ALTERNATIVAS = {'LPIPS': 'less', 'LPIPS_mask': 'less',
                'PSNR': 'greater', 'PSNR_mask': 'greater',
                'SSIM': 'greater', 'SSIM_mask': 'greater'}
ALFA = 0.05
M_COMPARACIONES = len(ALTERNATIVAS)          # familia de Bonferroni

f_pre = final[final['clave_config'] == 'pre'].set_index('imagen')
f_ft  = final[final['clave_config'] == 'ft'].set_index('imagen')
comunes = f_pre.index.intersection(f_ft.index)
n = len(comunes)

filas_wx = []
for met, alt in ALTERNATIVAS.items():
    v_pre = f_pre.loc[comunes, met].astype(float).values
    v_ft  = f_ft.loc[comunes,  met].astype(float).values
    ok = np.isfinite(v_pre) & np.isfinite(v_ft)
    v_pre, v_ft = v_pre[ok], v_ft[ok]
    dif = v_ft - v_pre
    mejora = dif < 0 if alt == 'less' else dif > 0

    if np.allclose(dif, 0):
        stat, p = np.nan, 1.0
    else:
        stat, p = wilcoxon(v_ft, v_pre, alternative=alt)
    p_adj = min(1.0, p * M_COMPARACIONES)

    filas_wx.append({
        'Métrica': met, 'n': int(ok.sum()), 'H₁': f'ajustado {"<" if alt=="less" else ">"} preentrenado',
        'Δ medio': float(dif.mean()), 'Δ mediana': float(np.median(dif)),
        'Mejoran': f'{int(mejora.sum())}/{int(ok.sum())}',
        'W': float(stat) if stat == stat else np.nan,
        'p': float(p), 'p Bonferroni': float(p_adj),
        'Significativo (α=0,05)': 'sí' if p_adj < ALFA else 'no',
    })

df_wx = pd.DataFrame(filas_wx)
print(f'Wilcoxon pareado, n = {n}, familia de {M_COMPARACIONES} comparaciones '
      f'(umbral corregido α = {ALFA/M_COMPARACIONES:.4f})\n')
print(df_wx.to_string(index=False, float_format='{:.4f}'.format))
df_wx.to_csv(DIR_METRICAS / 'wilcoxon_5_4.csv', index=False)

## 7. Figuras

### 7.1 Trayectoria de métricas a lo largo del pipeline

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 5.6), sharex=True)
paneles = [('LPIPS', 'LPIPS ↓'), ('PSNR', 'PSNR ↑ (dB)'), ('SSIM', 'SSIM ↑'),
           ('LPIPS_mask', 'LPIPS máscara ↓'), ('PSNR_mask', 'PSNR máscara ↑ (dB)'),
           ('SSIM_mask', 'SSIM máscara ↑')]
x = np.arange(len(ORDEN_ETAPAS))

for ax, (met, etiq) in zip(axes.ravel(), paneles):
    for cfg, color, marca in [('Preentrenado', OI_ORANGE, 'o'), ('Ajustado', OI_BLUE, 's')]:
        sub = df_met[df_met['config'] == cfg]
        medias = [sub[sub['etapa'] == e][met].mean() for e in ORDEN_ETAPAS]
        errs   = [sub[sub['etapa'] == e][met].std(ddof=1) / np.sqrt(len(df_conjunto))
                  for e in ORDEN_ETAPAS]
        ax.errorbar(x, medias, yerr=errs, color=color, marker=marca, markersize=4,
                    linewidth=1.4, capsize=3, label=cfg)
    ax.set_title(etiq)
    ax.set_xticks(x)
    ax.set_xticklabels(ORDEN_ETAPAS, rotation=12, ha='right')
    ax.grid(axis='y', alpha=0.25)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[0, 0].legend(frameon=False, loc='best')
fig.suptitle('Trayectoria de métricas por etapas del pipeline '
             f'(media ± error estándar, n = {len(df_conjunto)})',
             fontsize=9, fontweight='bold')
fig.text(0.5, -0.02,
         'La referencia se remuestrea al tamaño de cada predicción: las dos primeras etapas se '
         'miden a resolución LR y la tercera a ×4.\nLa comparación válida es entre '
         'configuraciones dentro de cada etapa, no entre etapas.',
         ha='center', fontsize=7, color='#444444')
plt.tight_layout()
ruta_fig = DIR_FIGURAS / 'trayectoria_metricas_5_4.png'
fig.savefig(ruta_fig, dpi=300, bbox_inches='tight')
plt.show()
print('Figura guardada:', ruta_fig)

### 7.2 Distribución de LPIPS del pipeline completo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))

for ax, met, etiq in [(axes[0], 'LPIPS', 'LPIPS ↓ (global)'),
                      (axes[1], 'LPIPS_mask', 'LPIPS ↓ (región de máscara)')]:
    v_pre = f_pre.loc[comunes, met].astype(float).values
    v_ft  = f_ft.loc[comunes,  met].astype(float).values
    datos = [v_pre[np.isfinite(v_pre)], v_ft[np.isfinite(v_ft)]]
    colores = [OI_ORANGE, OI_BLUE]

    bp = ax.boxplot(datos, patch_artist=True, widths=0.45,
                    medianprops=dict(color='white', linewidth=2),
                    whiskerprops=dict(linewidth=1.2), capprops=dict(linewidth=1.2),
                    flierprops=dict(marker='o', markersize=3, alpha=0.5))
    for patch, c in zip(bp['boxes'], colores):
        patch.set_facecolor(c); patch.set_alpha(0.75)

    rng = np.random.default_rng(42)
    for k, (vals, c) in enumerate(zip(datos, colores), start=1):
        ax.scatter(np.full(len(vals), k) + rng.uniform(-0.12, 0.12, size=len(vals)), vals,
                   color=c, edgecolors='white', linewidths=0.4, s=22, alpha=0.85, zorder=3)
    for a, b in zip(v_pre, v_ft):
        if np.isfinite(a) and np.isfinite(b):
            ax.plot([1, 2], [a, b], color='#aaaaaa', linewidth=0.5, alpha=0.4, zorder=2)

    p_adj = float(df_wx.loc[df_wx['Métrica'] == met, 'p Bonferroni'].iloc[0])
    y_max = max(np.nanmax(datos[0]), np.nanmax(datos[1]))
    y_ann = y_max + 0.015
    ax.plot([1, 1, 2, 2], [y_ann, y_ann + 0.005, y_ann + 0.005, y_ann], color='black', linewidth=1)
    ax.text(1.5, y_ann + 0.008,
            f'p = {p_adj:.3f}' + (' *' if p_adj < ALFA else ' n.s.'),
            ha='center', va='bottom', fontsize=8)

    ax.set_xticks([1, 2]); ax.set_xticklabels(['Preentrenado', 'Ajustado'])
    ax.set_ylabel(etiq)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

fig.suptitle(f'Pipeline completo — LPIPS por imagen (n = {n}, p con corrección de Bonferroni)',
             fontsize=9, fontweight='bold')
plt.tight_layout()
ruta_fig = DIR_FIGURAS / 'lpips_boxplot_pipeline_5_4.png'
fig.savefig(ruta_fig, dpi=300, bbox_inches='tight')
plt.show()
print('Figura guardada:', ruta_fig)

### 7.3 Validación cualitativa

Dos figuras. La primera recorre la cadena completa sobre cuatro imágenes: las dos en las que el
pipeline ajustado más mejora el LPIPS de la región de máscara y las dos en las que la diferencia
es menor, para no seleccionar solo casos favorables. La segunda amplía el recorte centrado en el
daño, que es donde se decide si la ampliación ×4 conserva o amplifica los artefactos del
inpainting.

In [ ]:
# Selección: 2 casos de mayor mejora en LPIPS de máscara + 2 de diferencia mínima.
delta_mask = (f_pre.loc[comunes, 'LPIPS_mask'].astype(float)
              - f_ft.loc[comunes, 'LPIPS_mask'].astype(float)).dropna()
mejores = delta_mask.nlargest(2).index.tolist()
neutros = delta_mask.drop(index=mejores).abs().nsmallest(2).index.tolist()
ejemplos = mejores + neutros
assert ejemplos, 'No hay ejemplos con LPIPS de máscara calculable.'
print('Ejemplos seleccionados (2 de mayor mejora + 2 de diferencia mínima):', ejemplos)

cabeceras = ['Entrada (LR + daño)', 'Enmascarada', 'Pipeline preentrenado',
             'Pipeline ajustado', 'Referencia HR']
fig, axes = plt.subplots(len(ejemplos), 5, figsize=(12.5, 2.5 * len(ejemplos)))
if len(ejemplos) == 1:
    axes = axes[np.newaxis, :]
for col, cab in enumerate(cabeceras):
    axes[0, col].set_title(cab, fontsize=8, fontweight='bold', pad=3)

for fila, stem in enumerate(ejemplos):
    imgs = [_leer_rgb(DIR_ENTRADA / f'{stem}.png'),
            _leer_rgb(DIR_ENMASCARADA / f'{stem}.png'),
            _leer_rgb(DIR_SR_PRE / f'{stem}.png'),
            _leer_rgb(DIR_SR_FT / f'{stem}.png'),
            _leer_rgb(df_conjunto.loc[stem, 'ruta_hr'])]
    for col, img in enumerate(imgs):
        axes[fila, col].imshow(img)
        axes[fila, col].set_xticks([]); axes[fila, col].set_yticks([])
        for sp in axes[fila, col].spines.values():
            sp.set_visible(False)
    for col, origen in [(2, f_pre), (3, f_ft)]:
        axes[fila, col].text(0.02, 0.03,
                             f'LPIPS={float(origen.loc[stem, "LPIPS"]):.3f}',
                             transform=axes[fila, col].transAxes, fontsize=7, color='white',
                             bbox=dict(facecolor='black', alpha=0.55, pad=1, edgecolor='none'))
    axes[fila, 0].text(0.02, 0.03, stem[:18], transform=axes[fila, 0].transAxes,
                       fontsize=7, color='white',
                       bbox=dict(facecolor='black', alpha=0.55, pad=1, edgecolor='none'))

plt.suptitle('Comparación cualitativa del pipeline completo — test sintético con referencia',
             fontsize=9, fontweight='bold', y=0.995)
plt.subplots_adjust(wspace=0.02, hspace=0.05, top=0.95, bottom=0.01, left=0.01, right=0.99)
ruta_fig = DIR_FIGURAS / 'cualitativa_pipeline_5_4.png'
fig.savefig(ruta_fig, dpi=200, bbox_inches='tight')
plt.show()
print('Figura guardada:', ruta_fig)

In [ ]:
# Detalle: recorte centrado en el daño, a resolución de salida (×4).
MARGEN = 0.35   # margen relativo alrededor del rectángulo envolvente de la máscara

def recorte_danio(img_rgb, mask_lr, escala_wh):
    w, h = escala_wh
    msk = _alinear(mask_lr, (w, h), interp=cv2.INTER_NEAREST) > 127
    bbox = _bbox_mascara(msk, lado_min=64)
    if bbox is None:
        ch, cw = int(h * 0.3), int(w * 0.3)
        y0, x0 = (h - ch) // 2, (w - cw) // 2
        return img_rgb[y0:y0 + ch, x0:x0 + cw]
    y0, y1, x0, x1 = bbox
    my, mx = int((y1 - y0) * MARGEN), int((x1 - x0) * MARGEN)
    y0, y1 = max(0, y0 - my), min(h, y1 + my)
    x0, x1 = max(0, x0 - mx), min(w, x1 + mx)
    return img_rgb[y0:y1, x0:x1]


cabeceras = ['Entrada (recorte)', 'Pipeline preentrenado', 'Pipeline ajustado', 'Referencia HR']
fig, axes = plt.subplots(len(ejemplos), 4, figsize=(10, 2.6 * len(ejemplos)))
if len(ejemplos) == 1:
    axes = axes[np.newaxis, :]
for col, cab in enumerate(cabeceras):
    axes[0, col].set_title(cab, fontsize=8, fontweight='bold', pad=3)

for fila, stem in enumerate(ejemplos):
    mask_lr = cv2.imread(str(DIR_MASCARA / f'{stem}.png'), cv2.IMREAD_GRAYSCALE)
    sr_pre  = _leer_rgb(DIR_SR_PRE / f'{stem}.png')
    sr_ft   = _leer_rgb(DIR_SR_FT / f'{stem}.png')
    hr      = _alinear(_leer_rgb(df_conjunto.loc[stem, 'ruta_hr']),
                       (sr_pre.shape[1], sr_pre.shape[0]))
    ent     = _alinear(_leer_rgb(DIR_ENTRADA / f'{stem}.png'),
                       (sr_pre.shape[1], sr_pre.shape[0]), interp=cv2.INTER_NEAREST)
    tam = (sr_pre.shape[1], sr_pre.shape[0])

    for col, img in enumerate([ent, sr_pre, sr_ft, hr]):
        axes[fila, col].imshow(recorte_danio(img, mask_lr, tam))
        axes[fila, col].set_xticks([]); axes[fila, col].set_yticks([])
        for sp in axes[fila, col].spines.values():
            sp.set_visible(False)
    for col, origen in [(1, f_pre), (2, f_ft)]:
        v = origen.loc[stem, 'LPIPS_mask']
        if np.isfinite(v):
            axes[fila, col].text(0.02, 0.03, f'LPIPS másc.={float(v):.3f}',
                                 transform=axes[fila, col].transAxes, fontsize=7, color='white',
                                 bbox=dict(facecolor='black', alpha=0.55, pad=1, edgecolor='none'))

plt.suptitle('Detalle de la región reconstruida — ¿conserva o amplifica la ampliación ×4 '
             'los artefactos del inpainting?', fontsize=9, fontweight='bold', y=0.995)
plt.subplots_adjust(wspace=0.02, hspace=0.05, top=0.95, bottom=0.01, left=0.01, right=0.99)
ruta_fig = DIR_FIGURAS / 'detalle_mascara_5_4.png'
fig.savefig(ruta_fig, dpi=200, bbox_inches='tight')
plt.show()
print('Figura guardada:', ruta_fig)

## 8. Exportación de resultados y síntesis

Todo lo que la memoria necesita citar queda escrito en `Fase5_4/metricas/` para no depender de
volver a ejecutar el notebook.

In [ ]:
resultados = {
    'apartado': '5.4 — Pipeline completo con modelos ajustados',
    'n_imagenes': int(n),
    'semilla_danio': SEED_BASE,
    'conjunto': {
        'origen_lr': str(DIR_TEST_LR),
        'origen_hr': str(DIR_TEST_HR),
        'danio': 'synthetic_degradation.DamageConfig.vintage_base() (mismo que 5.2)',
        'cobertura_media': float(df_conjunto['cobertura'].mean()),
        'cobertura_min': float(df_conjunto['cobertura'].min()),
        'cobertura_max': float(df_conjunto['cobertura'].max()),
    },
    'modelos': {
        'preentrenado': {'inpainting': 'big-lama', 'sr': str(PTH_AESRGAN_PRE.name)},
        'ajustado': {'inpainting': str(PTH_LAMA_FT.name),
                     'sr': f'brazo F_flick — {PTH_AESRGAN_FT.name}'},
    },
    'pipeline_completo': {
        cfg: {m: float(piv.loc[cfg, m]) for m in METRICAS} for cfg in CONFIGS.values()
    },
    'wilcoxon': {
        'alfa': ALFA,
        'correccion': f'Bonferroni, m = {M_COMPARACIONES}',
        'umbral_corregido': ALFA / M_COMPARACIONES,
        'tests': df_wx.to_dict(orient='records'),
    },
    'figuras': sorted(p.name for p in DIR_FIGURAS.glob('*.png')),
}

ruta_json = DIR_METRICAS / 'resultados_5_4.json'
ruta_json.write_text(json.dumps(resultados, indent=2, ensure_ascii=False), encoding='utf-8')
print('Resultados exportados:', ruta_json)

In [ ]:
print('=' * 72)
print('SÍNTESIS — Apartado 5.4: pipeline completo con referencia (OE3)')
print('=' * 72)
print(f'\nConjunto: {n} imágenes del test sintético de super-resolución con degradación '
      f'espectral\ncalibrada más el daño localizado del módulo de inpainting de la Fase 1 '
      f'(cobertura media {df_conjunto["cobertura"].mean():.2%}).\n')

for cfg in CONFIGS.values():
    print(f'{cfg:>13s}: LPIPS={piv.loc[cfg, "LPIPS"]:.4f}  '
          f'PSNR={piv.loc[cfg, "PSNR"]:.2f} dB  SSIM={piv.loc[cfg, "SSIM"]:.4f}   │   '
          f'máscara: LPIPS={piv.loc[cfg, "LPIPS_mask"]:.4f}  '
          f'PSNR={piv.loc[cfg, "PSNR_mask"]:.2f} dB  SSIM={piv.loc[cfg, "SSIM_mask"]:.4f}')

print(f'\nWilcoxon pareado con Bonferroni (m = {M_COMPARACIONES}, '
      f'umbral α = {ALFA/M_COMPARACIONES:.4f}):')
for _, r in df_wx.iterrows():
    print(f'  {r["Métrica"]:<11s} Δ medio={r["Δ medio"]:+.4f}  mejoran {r["Mejoran"]:>6s}  '
          f'p={r["p"]:.4f}  p_adj={r["p Bonferroni"]:.4f}  '
          f'{"✓ significativo" if r["Significativo (α=0,05)"] == "sí" else "✗ no significativo"}')

print('\nSalidas:')
print(f'  métricas → {DIR_METRICAS}')
print(f'  figuras  → {DIR_FIGURAS}')
for f in sorted(DIR_FIGURAS.glob("*.png")):
    print(f'    {f.name}')

print('\nRecordatorios para la redacción de 5.4:')
print('  · Estas cifras no son comparables con la línea base de 5.1 (protocolo de degradación')
print('    distinto). Decirlo en el texto cada vez que las dos columnas aparezcan juntas.')
print('  · La trayectoria por etapas cambia de escala de referencia entre LaMa y A-ESRGAN:')
print('    la comparación válida es entre configuraciones dentro de cada etapa.')
print('  · El checkpoint de A-ESRGAN se seleccionó contra el test (declarado en 5.3); la')
print('    cifra de este apartado hereda esa condición de cota superior.')
print('  · Nada de material histórico real en este apartado: eso es el capítulo 6.')

## 9. Notas de reproducibilidad

- El daño localizado se genera con `numpy.random.default_rng(SEED_BASE + i)`, donde `i` es el
  índice de la imagen en el listado ordenado alfabéticamente del directorio LR. Cambiar el
  contenido del directorio cambia los índices y por tanto el daño: si se reejecuta el notebook
  para reproducir cifras publicadas, el conjunto LR debe ser el mismo.
- El conjunto generado (`entrada/`, `mascaras/`, `enmascarada/`) queda escrito en Drive, así que
  la sección 2 puede saltarse en reejecuciones posteriores y las cifras se mantienen aunque el
  paquete de degradación cambie.
- Las salidas intermedias de LaMa se conservan (`lama_pre/`, `lama_ft/`) porque son la entrada de
  la etapa de super-resolución y permiten reejecutar solo la segunda mitad del pipeline.
- Versiones a fijar en el anexo: las que imprime la primera celda de entorno más
  `simple-lama-inpainting`, `basicsr`, `lpips` y `scikit-image`.